# Step 1 — Data Generation

**What this notebook does:** Generates synthetic lab data simulating Roche R&D laboratory experiments with realistic seasonality, machine degradation, and reagent batch quality.

**Inputs:** None (standalone generation)

**Outputs:**
- `../data/raw/workflow_logs.csv` (350K experiments)
- `../data/raw/telemetry_logs.csv` (350K telemetry rows)
- `../data/raw/reagent_logs.csv` (350K rows, 50 batches, 3 bad)

**Run order:** Run first (Step 1 of 5).

# Roche Capstone - Advanced Data Generation V2.0

This notebook generates complex, highly realistic synthetic data for the lab simulation. Unlike V1, this version uses **Structural Causality** instead of simple randomness.

### Key Features (The "Robust" Logic)
1. **Seasonality**: Non-Homogeneous Poisson Process for booking times (peaks at 10am/2pm, Mon-Wed).
2. **Machine Degradation**: Instruments lose 'health' over usage hours, increasing failure probability.
3. **Bad Batches**: Specific reagent batches cause cluster failures (5x risk).
4. **Probabilistic Delays**: Delays are drawn from a Gamma distribution based on a multi-factor Risk Score.

In [1]:
import pandas as pd
import numpy as np
import random
from scipy.stats import gamma

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

N_ROWS = 350000

### 1. The Time Engine (Seasonality)

In [2]:
def generate_seasonal_timestamps(n, start_date='2024-01-01', days=365):
    print("Generating Seasonal Timestamps...")
    
    # Base: Uniform random seconds over the year
    dates = pd.date_range(start=start_date, periods=days*24, freq='H') # Placeholder backbone
    
    # We simulate by rejection sampling or weighted choice. Weighted choice is faster for N=350k.
    
    # 1. Select Day of Year based on Weekly Cycle
    # Weights: Mon(0)-Sun(6)
    # Mon(4), Tue(4), Wed(4), Thu(3), Fri(2), Sat(0.5), Sun(0.5)
    day_weights = {0:4, 1:4, 2:4, 3:3, 4:2, 5:0.5, 6:0.5}
    
    all_days = pd.date_range(start_date, periods=days, freq='D')
    day_probs = [day_weights[d.dayofweek] for d in all_days]
    day_probs = np.array(day_probs) / sum(day_probs)
    
    chosen_days = np.random.choice(all_days, n, p=day_probs)
    
    # 2. Select Hour of Day based on Daily Cycle
    # Peaks: 10am, 2pm. Low: Lunch(12), Night(20-06)
    hours = np.arange(24)
    # Heuristic weight curve
    hour_weights = [
        0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.5, 2, 4, 3, 5, 4, # 00-11 (Peak 10)
        2, 3, 5, 4, 3, 2, 1, 0.5, 0.2, 0.1, 0.1, 0.1      # 12-23 (Lunch dip, Peak 14)
    ]
    hour_probs = np.array(hour_weights) / sum(hour_weights)
    chosen_hours = np.random.choice(hours, n, p=hour_probs)
    
    # Add random minutes/seconds
    timestamps = []
    for d, h in zip(chosen_days, chosen_hours):
        ts = d + pd.Timedelta(hours=h, minutes=np.random.randint(0, 60), seconds=np.random.randint(0, 60))
        timestamps.append(ts)
        
    return pd.Series(timestamps).sort_values().reset_index(drop=True)

### 2. Core Data Generation

In [3]:
# --- Setup ---
timestamps = generate_seasonal_timestamps(N_ROWS)
experiment_types = ['Validation', 'QC', 'Pilot', 'Screening', 'R&D']
inst_map = {
    'Validation': ['Microscope', 'Centrifuge'], 'QC': ['Spectrometer', 'HPLC'],
    'Pilot': ['Incubator', 'PCR'], 'Screening': ['PCR', 'Centrifuge'], 'R&D': ['HPLC', 'Incubator']
}

# --- Assignments ---
exp_types = np.random.choice(experiment_types, N_ROWS, p=[0.25, 0.25, 0.2, 0.2, 0.1])
inst_types = [random.choice(inst_map[et]) for et in exp_types]

# --- Instrument ID Assignment (Critical for Degradation) ---
# Assume 20 instruments total
n_instruments = 20
instrument_ids_pool = [f'INST_{i:03d}' for i in range(n_instruments)]
# Deterministic mapping based on type to ensure consistency? Or just random pool?
# Simpler: Each instrument has a fixed type. 
inst_inventory = []
for itype in set([i for sublist in inst_map.values() for i in sublist]):
    for k in range(5): # 5 of each type
        inst_inventory.append({'id': f'{itype}_{k:02d}', 'type': itype, 'health': 1.0, 'hours_used': 0})
inst_df = pd.DataFrame(inst_inventory)

# Assign an instrument_id available for that type
# This approximation randomly picks valid instrument for the row
assigned_inst_ids = []
for itype in inst_types:
    valid_ids = inst_df[inst_df['type'] == itype]['id'].values
    assigned_inst_ids.append(np.random.choice(valid_ids))

# --- Reagent Batches ---
batches = [f'BATCH_{i:03d}' for i in range(50)]
BAD_BATCHES = np.random.choice(batches, 3, replace=False)
print(f"Selected Bad Batches: {BAD_BATCHES}")
reagent_batch_ids = np.random.choice(batches, N_ROWS)

# Create DataFrame
df = pd.DataFrame({
    'experiment_id': [f'EXP_{i:06d}' for i in range(N_ROWS)],
    'booking_time': timestamps,
    'experiment_type': exp_types,
    'instrument_type': inst_types,
    'instrument_id': assigned_inst_ids,
    'reagent_batch_id': reagent_batch_ids,
    'scientist_workload': np.random.poisson(5, N_ROWS).clip(1, 15),
    'scientist_experience_level': np.random.choice(['Junior', 'Mid', 'Senior'], N_ROWS, p=[0.4, 0.4, 0.2]),
    'lab_occupancy_level': np.random.normal(62, 18, N_ROWS).clip(0, 100).astype(int)
})

# Expected Durations
duration_map = {'Validation': 60, 'QC': 45, 'Pilot': 90, 'Screening': 30, 'R&D': 120}
df['expected_duration'] = df['experiment_type'].map(duration_map)

Generating Seasonal Timestamps...


/var/folders/80/vx7bsx9917v8fw_1j_dc367h0000gn/T/ipykernel_77259/3533139293.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start=start_date, periods=days*24, freq='H') # Placeholder backbone


Selected Bad Batches: ['BATCH_044' 'BATCH_027' 'BATCH_022']


### 3. Calculating Machine Degradation (The Drift)
We simulate the chronological usage of machines. Every hour of use slightly damages the machine.

In [4]:
# --- 3. Calculating Machine Degradation (Reliability, not Slowing Down) ---

df = df.sort_values('booking_time').reset_index(drop=True)

health_scores = []
failure_flags = []

# track cumulative usage minutes per instrument
usage_minutes = {inst_id: 0.0 for inst_id in inst_df['id']}

# instrument-type baseline failure rate (per run, before aging)
base_fail = {
    'PCR': 0.010,
    'HPLC': 0.008,
    'Incubator': 0.006,
    'Spectrometer': 0.006,
    'Centrifuge': 0.005,
    'Microscope': 0.003
}

print("Simulating Machine Reliability Degradation...")
for _, row in df.iterrows():
    iid = row['instrument_id']
    itype = row['instrument_type']
    dur = float(row['expected_duration'])  # minutes

    # accumulate usage
    usage_minutes[iid] += dur

    # wear index grows with usage (0 → ~1 over time); slow early, faster later
    wear = min(1.0, (usage_minutes[iid] / 120000.0) ** 1.6)  # 120k min ~ 2000 hours

    # health: stays high for long time; only drops meaningfully late-life
    health = max(0.2, 1.0 - 0.75 * wear)

    # failure probability increases with wear (nonlinear)
    p0 = base_fail.get(itype, 0.006)
    p_fail = min(0.20, p0 * (1 + 6 * wear**2))  # ramps up near end-of-life

    failed = (np.random.rand() < p_fail)

    # if a failure happens, health takes a hit (represents breakdown / recalibration event)
    if failed:
        health = max(0.05, health - 0.25)

    health_scores.append(health)
    failure_flags.append(int(failed))

df['instrument_health'] = health_scores
df['machine_failure'] = failure_flags

print(f"Avg health: {df['instrument_health'].mean():.3f} | Failure rate: {df['machine_failure'].mean():.3%}")

Simulating Machine Reliability Degradation...
Avg health: 0.322 | Failure rate: 4.037%


### 4. The Probabilistic Output Engine
Calculating `Delay_Risk` and Gamma-distributed delays.

In [5]:
# --- 1. Temperature Simulation (Instrument-Specific Logic) ---

# Setpoint per instrument (realistic lab assumptions)
setpoint_map = {
    'Incubator': 37,       # biological growth temp
    'PCR': 95,             # high thermal cycling
    'HPLC': 25,            # controlled room temp
    'Spectrometer': 22,    # sensitive optics
    'Microscope': 22,
    'Centrifuge': 20
}

df['temp_setpoint'] = df['instrument_type'].map(setpoint_map)

# Ambient lab room temperature (can drift)
df['mean_ambient_temp'] = np.random.normal(23, 3, N_ROWS)

# Deviation from required setpoint (not from fixed 22C)
df['temp_deviation'] = abs(df['mean_ambient_temp'] - df['temp_setpoint'])

# Nonlinear temperature risk (small deviations ok, large ones dangerous)
norm_temp = np.clip(df['temp_deviation'] / 20, 0, 1) ** 1.7

# --- 2. Risk Calculation ---
# Normalize factors to 0-1 scale approximately
occ = df['lab_occupancy_level'].astype(float)

# Logistic: slow increase then sharp rise near the tipping region (~75-80)
crowding_risk = 1 / (1 + np.exp(-(occ - 75) / 5))  # center=75, steepness=5
norm_wear = 1.0 - df['instrument_health'] # 0 is new, 1 is dead
norm_temp = np.clip(df['temp_deviation'] / 15, 0, 1) ** 1.6

risk_score = (0.45 * crowding_risk) + (0.30 * norm_wear) + (0.25 * norm_temp)

# --- 3. Bad Batch Multiplier ---
batch_mask = df['reagent_batch_id'].isin(BAD_BATCHES)
risk_score[batch_mask] *= 5.0

# --- 4. Gamma Delay ---
df['delay'] = np.random.gamma(shape=(risk_score * 12) + 0.5, scale=5.5)

# 🔥 ADD MACHINE FAILURE EFFECT HERE (RIGHT AFTER DELAY)

# Machine failures create special incidents + extra delay (recalibration / rerun)
df.loc[df['machine_failure'] == 1, 'delay'] += np.random.gamma(
    shape=4, scale=10, size=(df['machine_failure'] == 1).sum()
)

# --- 5. Clean Up ---
df['incident_type'] = 'None'

# First: machine failures override everything
df.loc[df['machine_failure'] == 1, 'incident_type'] = 'Machine_Failure'

# Then other logic
df.loc[(df['delay'] > 30) & (df['incident_type'] == 'None'), 'incident_type'] = 'Resource_Contention'
df.loc[(df['delay'] > 60) & (df['incident_type'] == 'None'), 'incident_type'] = 'Instrument_Failure'
df.loc[batch_mask & (df['delay'] > 45) & (df['incident_type'] == 'None'), 'incident_type'] = 'Reagent_Quality'

# Actual Duration
df['actual_duration'] = df['expected_duration'] + df['delay']

display(df[['booking_time', 'instrument_health', 'reagent_batch_id', 'delay']].head(10))

,booking_time,instrument_health,reagent_batch_id,delay
0,2024-01-01 00:18:04,0.999992,BATCH_013,8.944742
1,2024-01-01 00:44:22,0.999998,BATCH_016,0.895662
2,2024-01-01 01:05:07,0.999998,BATCH_005,0.618095
3,2024-01-01 01:21:09,0.999998,BATCH_046,18.556023
4,2024-01-01 01:42:56,0.999999,BATCH_034,40.318977
5,2024-01-01 02:32:41,0.999999,BATCH_005,18.938264
6,2024-01-01 02:38:47,0.999996,BATCH_032,0.851942
7,2024-01-01 02:52:22,0.999977,BATCH_046,24.922687
8,2024-01-01 04:05:38,0.999988,BATCH_019,9.011872
9,2024-01-01 04:27:58,0.999996,BATCH_044,2.858242


In [6]:
# Save Files
# Workflow Logs
df_wk = df[['experiment_id', 'experiment_type', 'instrument_type', 'instrument_id', 
            'scientist_workload', 'scientist_experience_level', 'lab_occupancy_level', 
            'expected_duration', 'booking_time', 'actual_duration', 'delay', 'incident_type', 'machine_failure', 'instrument_health']].copy()

# Reagent Logs
df_rg = df[['experiment_id', 'reagent_batch_id', 'booking_time']].copy()

# Telemetry Logs (Simulated aggregate for now to match V1 schema)
df_tl = df[['experiment_id', 'mean_ambient_temp', 'temp_setpoint', 'temp_deviation']].copy()
df_tl['ambient_temp'] = df_tl['mean_ambient_temp'] # simplified
df_tl = df_tl.drop('mean_ambient_temp', axis=1)
df_tl['timestamp'] = df['booking_time'] # simplified

df_wk.to_csv('../data/raw/workflow_logs.csv', index=False)
df_rg.to_csv('../data/raw/reagent_logs.csv', index=False)
# Telemetry needs to be long format? For now keeping 1-to-1 to match merger logic in notebook 2/3
df_tl.to_csv('../data/raw/telemetry_logs.csv', index=False)

print("Data Generation V2 Complete. Saved csvs.")

Data Generation V2 Complete. Saved csvs.


In [7]:
sorted(df['instrument_type'].unique())

['Centrifuge', 'HPLC', 'Incubator', 'Microscope', 'PCR', 'Spectrometer']

In [8]:
df['instrument_type'].value_counts()

instrument_type
Centrifuge      78972
PCR             70071
HPLC            60914
Incubator       52860
Spectrometer    43749
Microscope      43434
Name: count, dtype: int64

In [9]:
print("Failure rate:", df['machine_failure'].mean())
print(df['incident_type'].value_counts().head())
print(df.groupby('machine_failure')['delay'].mean())

Failure rate: 0.04037142857142857
incident_type
None                   182119
Resource_Contention    153751
Machine_Failure         14130
Name: count, dtype: int64
machine_failure
0    36.145026
1    84.936372
Name: delay, dtype: float64


In [10]:
df.groupby('instrument_type')['machine_failure'].mean().sort_values(ascending=False)

instrument_type
PCR             0.062023
HPLC            0.049168
Incubator       0.038630
Spectrometer    0.034195
Centrifuge      0.031340
Microscope      0.017866
Name: machine_failure, dtype: float64